# nb02 preprocessing

LunarAI — SIH 26166 notebook. Generated by build_notebooks.py.

## 1. Objective
Apply the standardization pipeline (grayscale, CLAHE, histogram equalization, contrast
normalization, noise reduction, resize) to every usable image, producing the enhanced
imagery used for patch extraction in notebook 03.

## 2. Dataset Loading

In [1]:
import sys
from pathlib import Path

def _find_project_root() -> Path:
    # anchor at the directory that actually contains the project markers,
    # independent of the kernel's working directory
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "lunarai_lib").is_dir() and (cand / "configs").is_dir():
            return cand
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from lunarai_lib.config import load_config
from lunarai_lib import metadata as md
from lunarai_lib import preprocessing as pp
from lunarai_lib.io_utils import load_image_any, load_browse_png, save_json, image_stats
import cv2
import numpy as np

cfg = load_config()
records_by_sensor = md.discover_all(cfg.DATA_ROOT)
all_records = [r for recs in records_by_sensor.values() for r in recs]
print(f"{len(all_records)} products loaded")

15 products loaded


## 3. Configuration

In [2]:
PROCESSED_DIR = cfg.PROCESSED_DIR
RESIZE_LONG = cfg.RESIZE_LONG_SIDE
PRE = cfg.PRE
print("CLAHE clip", PRE.get("clahe_clip_limit"), "| grid", PRE.get("clahe_grid"),
      "| denoise", PRE.get("denoise"))

CLAHE clip 2.0 | grid 8 | denoise median


## 4. Implementation

In [3]:
def preprocess_record(record, processed_dir):
    """Returns dict with paths + before/after stats, or None if unusable."""
    img = None
    source = "binary"
    if record.binary_present:
        img = load_image_any(record, RESIZE_LONG)
    if img is None:
        img = load_browse_png(record, RESIZE_LONG)
        source = "browse_png_fallback"
    if img is None:
        return None

    enhanced = pp.full_preprocess(
        img,
        clahe_clip=float(PRE.get("clahe_clip_limit", 2.0)),
        clahe_grid=int(PRE.get("clahe_grid", 8)),
        denoise=str(PRE.get("denoise", "median")),
        denoise_ksize=int(PRE.get("denoise_ksize", 3)),
    )
    out_dir = processed_dir / record.sensor.replace(" ", "")
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = record.image_id.rsplit(".", 1)[0][:60]
    out_path = out_dir / f"{stem}_proc.png"
    cv2.imwrite(str(out_path), enhanced)
    before, after = image_stats(img), image_stats(enhanced)
    if source == "browse_png_fallback":
        record.notes += "preprocessed from browse PNG; "
    return {"image_id": record.image_id, "sensor": record.sensor, "source": source,
            "processed_path": str(out_path),
            "contrast_before": round(before["std"], 2), "contrast_after": round(after["std"], 2),
            "mean_before": round(before["mean"], 2), "mean_after": round(after["mean"], 2)}

results = []
for r in all_records:
    res = preprocess_record(r, PROCESSED_DIR)
    if res:
        results.append(res)
        print(f"  [{res['sensor']:7s}] {res['image_id'][:44]:44s} "
              f"contrast {res['contrast_before']:6.1f} -> {res['contrast_after']:6.1f}")
    else:
        print(f"  [{r.sensor:7s}] {r.image_id[:44]:44s} SKIPPED (no decodable image)")

  [OHRC   ] ch2_ohr_ncp_20241115T1326321339_d_img_d18.im contrast   42.7 ->   70.0


  [OHRC   ] ch2_ohr_ncp_20241115T1525004388_d_img_d18.im contrast   50.5 ->   73.4


  [OHRC   ] ch2_ohr_ncp_20251010T0942085687_d_img_d18.im contrast   46.7 ->   79.2


  [TMC-2  ] ch2_tmc_nca_20240124T0838058366_d_img_d18.im contrast   27.3 ->   67.5


  [TMC-2  ] ch2_tmc_ncf_20211122T1925079181_d_img_d18.im contrast   25.7 ->   64.1


  [TMC-2  ] ch2_tmc_ncf_20211122T2123225722_d_img_d18.im contrast   26.8 ->   67.4


  [TMC-2  ] ch2_tmc_ncf_20211122T2321361810_d_img_d18.im contrast   25.9 ->   67.5


  [TMC-2  ] ch2_tmc_ncf_20260701T0716265753_d_img_d18.im contrast   16.9 ->   53.9


  [TMC-2  ] ch2_tmc_ncn_20211122T1925079214_d_img_d18.im contrast   13.7 ->   58.1


  [TMC-2  ] ch2_tmc_ncn_20211123T0119537815_d_img_d18.im contrast   15.6 ->   61.2
  [IIRS   ] ch2_iir_nri_20210126T1349194455_d_img_d32.hd contrast   50.3 ->   62.4
  [IIRS   ] ch2_iir_nri_20210126T1545362143_d_img_d32.hd SKIPPED (no decodable image)
  [IIRS   ] ch2_iir_nri_20210126T1743334783_d_img_d32.hd SKIPPED (no decodable image)
  [IIRS   ] ch2_iir_nri_20240124T1430209615_d_img_d18.hd contrast   41.3 ->   58.8
  [LRO NAC] quickmap-lroc.png                            contrast   36.8 ->   59.7


## 5. Visualization — enhancement effect on a sample image

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sample = next((r for r in all_records if r.binary_present and r.sensor == "TMC-2"), all_records[0])
raw = load_image_any(sample, RESIZE_LONG)
enh = pp.full_preprocess(raw, clahe_clip=float(PRE.get("clahe_clip_limit", 2.0)),
                         clahe_grid=int(PRE.get("clahe_grid", 8)),
                         denoise=str(PRE.get("denoise", "median")),
                         denoise_ksize=int(PRE.get("denoise_ksize", 3)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(raw, cmap="gray"); axes[0].set_title("Raw")
axes[1].imshow(enh, cmap="gray"); axes[1].set_title("CLAHE + denoise + normalize")
axes[2].hist(raw.ravel(), bins=64, alpha=0.6, label="raw", color="#94A3B8")
axes[2].hist(enh.ravel(), bins=64, alpha=0.6, label="processed", color="#38BDF8")
axes[2].legend(); axes[2].set_title("Intensity histograms")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(cfg.VISUALIZATIONS_DIR / "preprocessing_comparison.png", dpi=120)
plt.show()

C:\Users\girid\AppData\Local\Temp\ipykernel_15616\50948041.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Metrics

In [5]:
contrast_gain = [r["contrast_after"] / max(r["contrast_before"], 1e-6) for r in results]
metrics = {
    "images_processed": len(results),
    "images_skipped": len(all_records) - len(results),
    "mean_contrast_gain": round(float(np.mean(contrast_gain)), 3) if contrast_gain else 0.0,
    "fallback_to_browse_png": sum(1 for r in results if r["source"] == "browse_png_fallback"),
}
print(metrics)

{'images_processed': 13, 'images_skipped': 2, 'mean_contrast_gain': 2.348, 'fallback_to_browse_png': 2}


## 7. Saved Outputs
- `outputs/processed/{OHRC,TMC-2,IIRS,LRONAC}/*_proc.png`
- `outputs/visualizations/preprocessing_comparison.png`

In [6]:
save_json({"metrics": metrics, "records": results},
          cfg.METRICS_DIR / "preprocessing_report.json")

WindowsPath('D:/lunar ai/outputs/metrics/preprocessing_report.json')

## 8. Error Handling
- Records without any decodable binary (IIRS without binaries) fall back to the browse PNG.
- If both fail, the record is skipped and counted in `images_skipped`.
- Extremely dark/flat images still pass through; contrast normalization clips to percentiles.

## 9. Explanation
The pipeline order is grayscale → CLAHE (adaptive local contrast for low-sun imagery) →
median denoise (removes salt-and-pepper sensor noise) → percentile contrast normalization
(harmonizes exposure across missions) → optional resize. CLAHE before denoise avoids
amplifying noise; percentile normalization is robust to crater shadows saturating the range.

## 10. Next-Step Integration
Notebook 03 extracts 256x256 patches from the *enhanced* images produced here, keeping the
raw-patch store and the processed-patch store aligned by filename.